## Hybrid Search

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
api_key = os.getenv("PINECONE_API_KEY")

In [6]:
from pinecone import Pinecone, ServerlessSpec
index_name = "hybrid-search-langchain"

# Initialize the Pinecone Client
pc = Pinecone(api_key=api_key)

# Create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, # Dimension of Dense Vector
        metric='dotproduct', # Sparse Value supported only for dotproduct
        spec=ServerlessSpec(cloud='aws', region="us-east-1")
    )

In [7]:
index = pc.Index(index_name)
index

Index(host='https://hybrid-search-langchain-5n7r5go.svc.aped-4627-b74a.pinecone.io')

In [ ]:
# Vector embedding and sparse matrix
from langchain_huggingface import HuggingFaceEmbeddings

hf_token = "xxx-xxxx-xxx"

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9831.89it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [18]:
from pinecone_text.sparse import BM25Encoder

encoder = BM25Encoder().default()
encoder

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/rishaan/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rishaan/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [40]:
sentences = [
    "In 2023, I visited Nepal.",
        "In 2022, I visited Bhutan.",
        "In 2021, I visited London."
]

encoder.fit(sentences)

# Store the values to a JSON file
encoder.dump("encoded_values.json")

encoder = BM25Encoder().load("encoded_values.json")

100%|██████████| 3/3 [00:00<00:00, 4782.56it/s]


In [41]:
dense_embeddings = embeddings.embed_documents(sentences)

sparse_embeddings = encoder.encode_documents(sentences)

In [42]:
vectors = []

for i, (text, dense, sparse) in enumerate(
    zip(sentences, dense_embeddings, sparse_embeddings)
):

    # Make sparse values float
    sparse["values"] = [
        float(value)
        for value in sparse["values"]
    ]

    vectors.append({
        "id": str(i),
        "values": dense,
        "sparse_values": sparse,
        "metadata": {
            "context": text
        }
    })

In [44]:
response = index.upsert(vectors=vectors)

In [46]:
query = "Which country I visited first?"

query_dense = embeddings.embed_query(query)

query_sparse = encoder.encode_queries([query])[0]

query_sparse["values"] = [
    float(value) for value in query_sparse["values"]
]

results = index.query(
    vector=query_dense,
    sparse_vector=query_sparse,
    top_k=3,
    include_metadata=True
)

print(results)

QueryResponse(matches=[ScoredVector(id='1', score=0.57582742, values=[], metadata={'context': 'In 2022, I visited Bhutan.'}), ScoredVector(id='2', score=0.535301387, values=[], metadata={'context': 'In 2021, I visited London.'}), ScoredVector(id='0', score=0.514339805, values=[], metadata={'context': 'In 2023, I visited Nepal.'})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Tue, 06 Oct 2026 22:50:45 GMT', 'content-type': 'application/json', 'content-length': '366', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '6', 'x-pinecone-request-latency-ms': '6', 'x-envoy-upstream-service-time': '5', 'x-pinecone-response-duration-ms': '7', 'grpc-status': '0', 'server': 'envoy'}))
